# Iteration 08 - No Complex Neurological Patients

## Aim

Analyse the impact of complex neurological patients on delay in the stroke pathway by comparing the current-admissions scenario with a scenario where complex neurological patients are excluded from the model.


## Prompt Used

In iteration 8, analyse the impact of complex neurological patients on delays in the stroke pathway. Model the likelihood of delay when there are no complex neurological patients, using the admission and transfer data reported in Monks et al. Present the results of the simulation in a table similar to the one pasted here, which is taken from the 'supplementary results' section of their appendix.


In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from stroke_sim.config import SimulationSettings
from stroke_sim.runner import run_replications
from stroke_sim.validation import (
    PUBLISHED_CURRENT_ADMISSIONS_ACUTE,
    PUBLISHED_CURRENT_ADMISSIONS_REHAB,
    PUBLISHED_NO_COMPLEX_ACUTE,
    PUBLISHED_NO_COMPLEX_REHAB,
    build_two_scenario_table,
    compare_to_published_table,
)


## Scenario Runs

We compare two scenarios:

- current admissions
- no complex neurological patients


In [ ]:
base_kwargs = dict(run_length_days=365 * 5, warm_up_days=365 * 3, replications=30)
current_audit = run_replications(settings=SimulationSettings(**base_kwargs))
no_complex_audit = run_replications(
    settings=SimulationSettings(excluded_patient_groups=('complex_neurological',), **base_kwargs)
)


## Acute Comparison Table


In [ ]:
acute_current = compare_to_published_table(
    current_audit,
    column='acute_occupancy',
    published=PUBLISHED_CURRENT_ADMISSIONS_ACUTE.rename(columns={'published_p_delay': 'published_p_delay'}),
)
acute_no_complex = compare_to_published_table(
    no_complex_audit,
    column='acute_occupancy',
    published=PUBLISHED_NO_COMPLEX_ACUTE,
)
acute_table = build_two_scenario_table(
    acute_current,
    acute_no_complex,
    bed_label='No. acute beds',
    baseline_prefix='current',
    scenario_prefix='no_complex',
)
acute_table


## Rehab Comparison Table


In [ ]:
rehab_current = compare_to_published_table(
    current_audit,
    column='rehab_occupancy',
    published=PUBLISHED_CURRENT_ADMISSIONS_REHAB,
)
rehab_no_complex = compare_to_published_table(
    no_complex_audit,
    column='rehab_occupancy',
    published=PUBLISHED_NO_COMPLEX_REHAB,
)
rehab_table = build_two_scenario_table(
    rehab_current,
    rehab_no_complex,
    bed_label='No. rehab beds',
    baseline_prefix='current',
    scenario_prefix='no_complex',
)
rehab_table


## Combined Supplementary Results Style Table


In [ ]:
import pandas as pd

supplementary_table = pd.concat([acute_table, rehab_table], ignore_index=True)
supplementary_table


## Short Summary


In [ ]:
summary = {
    'acute_current_mean_abs_error': round(float(acute_current['abs_error'].mean()), 4),
    'acute_no_complex_mean_abs_error': round(float(acute_no_complex['abs_error'].mean()), 4),
    'rehab_current_mean_abs_error': round(float(rehab_current['abs_error'].mean()), 4),
    'rehab_no_complex_mean_abs_error': round(float(rehab_no_complex['abs_error'].mean()), 4),
}
summary


## Tester Notes

- Complex neurological patients are excluded by removing their arrival streams from both acute admissions and transfers from elsewhere to rehab.
- Delay estimates continue to use the same Erlang-loss-style occupancy calculation that matched the paper best in previous iterations.
- The resulting table is shaped to resemble the supplementary results table in Monks et al.
